# 04 — Backtesting

Convert out-of-sample walk-forward volatility forecasts into a **volatility-targeting** strategy (scale BTC exposure inversely to forecast vol) and compare against buy-and-hold.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from coinpredictor.data.ohlcv import load_ohlcv
from coinpredictor.features import build_features
from coinpredictor.model import build_vol_regressor
from coinpredictor.backtest import walk_forward_backtest

feats = build_features(load_ohlcv())
result = walk_forward_backtest(build_vol_regressor, feats, fee=0.001)
print(result.summary())

In [ ]:
ax = result.equity.plot(figsize=(10, 5), title='Out-of-sample equity (growth of $1)')
ax.set_ylabel('Equity')

# Exposure over time: the strategy de-risks when it forecasts turbulence
result.weights.tail(365).plot(figsize=(10, 3), title='BTC exposure (volatility-target weight)')

## Interpreting results

- The goal of vol-targeting is **better Sharpe and smaller max drawdown**, not beating buy-and-hold on raw return (BTC's strong uptrend makes total return a tough benchmark).
- `forecast_corr` shows whether higher predicted vol actually precedes bigger moves — the core signal.
- Tune `STRATEGY.target_annual_vol` / `max_weight` in `config.py`, the `fee`, and add Phase 2/3 features (`build_features_full`) to see their effect.